In [ ]:
import time
import json
from spark_session_config import spark_cars
from pyspark.sql import functions as F
from kafka import KafkaProducer

cars_df = spark_cars.read.parquet("s3a://pyspark/data/dims/cars")

cars_sensor_df = cars_df \
    .withColumn("event_id", F.expr("uuid()")) \
    .withColumn("event_time", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("speed",(F.rand() * 201).cast("int")) \
    .withColumn("rpm",(F.rand() * 8001).cast("int")) \
    .withColumn("gear",(F.rand() * 7 + 1).cast("int")) \
    .select("event_id","event_time","car_id","speed","rpm","gear")

producer = KafkaProducer(bootstrap_servers='course-kafka:9092', value_serializer=lambda v: json.dumps(v).encode("utf-8"))

try:
    while True:
        for json_data in cars_sensor_df.toLocalIterator():
            producer.send(topic = 'sensors-sample',value = json_data.asDict())
        producer.flush()
        time.sleep(1)
finally:
    producer.close()
    spark_cars.stop()
